In [1]:
# imports

import pandas as pd
from dash import Dash, html, dcc, callback, Output, Input
import dash_ag_grid as dag
import plotly.express as px

In [2]:
# import patient data

df = pd.read_csv("./patients.csv")

df['last_visit'] = pd.to_datetime(df['last_visit'], format='%m/%d/%y').dt.date
df['followup'] = pd.to_datetime(df['followup'], format='%m/%d/%y').dt.date
df['first_name'] = df['first_name'].str.capitalize()
df['last_name'] = df['last_name'].str.capitalize()

df

,id,first_name,last_name,sex,dx,last_visit,followup,deceased
0,322,Joe,Marx,m,sick,2025-01-01,2026-05-01,n
1,326,Jane,Doe,f,cholera,2025-01-02,2026-05-02,n
2,975,Johnny,Cash,m,alcoholism,2025-01-11,2026-05-03,n
3,1278,Suzy,Hudson,f,pneumonia,2025-01-04,2026-05-04,n
4,599,Sam,Seuss,m,psychosis,2025-01-05,2026-05-05,n
5,370,Susan,Stevens,f,vocal injury,2025-01-19,2026-05-06,n
6,398,Hudson,Williams,m,sick,2025-01-09,2026-05-07,n
7,307,Connor,Storie,m,very hot,2025-01-08,2026-05-08,n
8,1053,Shane,Hollander,m,autism,2025-01-09,2026-05-09,n
9,386,Ilya,Razanov,m,very hot,2025-01-08,2026-05-10,n


# changing dates to datetime format; checks actually works on first row

print(type(df["last_visit"][0]))
df['last_visit'] = pd.to_datetime(df['last_visit'], format='%m/%d/%y')
print(type(df["last_visit"][0]))

print(type(df["followup"][0]))
df['followup'] = pd.to_datetime(df['followup'], format='%m/%d/%y')
print(type(df["followup"][0]))

# app with only data in a table

# Initialize the app
app = Dash()

# App layout
app.layout = [
    html.Div(children='Basic Patient Data'),
    dag.AgGrid(
        rowData=df.to_dict('records'),
        columnDefs=[{"field": i} for i in df.columns]
    )
]

# Run the app
if __name__ == '__main__':
    app.run(debug=True)

# app with table and histogram

# Initialize the app
app = Dash()

# App layout
app.layout = [
    html.Div(children='Basic Patient Data'),
    dag.AgGrid(
        rowData=df.to_dict('records'),
        columnDefs=[{"field": i} for i in df.columns]
    ),
    dcc.Graph(figure=px.histogram(df, x='dx', y='dx', histfunc='count'))
]

# dcc.Graph(figure=px.histogram(df, x='continent', y='lifeExp', histfunc='sum'))

# Run the app
if __name__ == '__main__':
    app.run(debug=True)

In [ ]:
# INTERACTIVE app with table and histogram

# Initialize the app
app = Dash()

# App layout
app.layout = [
    html.Div(children='Basic Patient Data'),
    html.Hr(),
    dcc.RadioItems(options=['dx', 'last_name', 'sex'], value='dx', id='controls-and-radio-item'),
    dag.AgGrid(
        rowData=df.to_dict('records'),
        columnDefs=[{"field": i} for i in df.columns]
    ),
    dcc.Graph(figure={}, id='current_graph'),
    dcc.Dropdown(value=[], id = 'drop_down_box')
]

# Add controls to build the interaction
@callback(
    Output(component_id='current_graph', component_property = 'figure'),
    # Output(component_id='drop_down_box', component_property = 'value'),      ##### trying to get this to fill in a dropdown box, so I can select "anxiety" and get a count for number of patients; then later add one for "due for followup", choose a date, and show everything after that date.
    Input(component_id='controls-and-radio-item', component_property = 'value')
)
def update_graph(col_chosen):
    ordered_axis = sorted(df[col_chosen].unique())
    fig = px.histogram(df, x = col_chosen, y = col_chosen, histfunc = 'count', category_orders = {col_chosen : ordered_axis}, text_auto = True)
    fig.update_layout(yaxis_title = f"{col_chosen} Total")
    return fig

# Run the app
if __name__ == '__main__':
    app.run(debug=True)